# Task 3 — Course Recommendation Engine

**Three signals:**
- **Markov:** P(next_module | current_module, last_outcome) — pathway-based
- **VLE-CF:** Pearson similarity on within-module activity profiles — behaviour-based
- **Content:** cosine similarity on 6-dim student–course feature match — profile-based

**Two evaluation populations:**
- Returning students (2013 → 2014): tests the full hybrid
- Brand-new 2014 students (2014B → 2014J): tests cold-start only

In [1]:
import sys, os, glob
sys.path.insert(0, os.path.abspath('.'))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_DIR   = '..'
OUTPUT_DIR = 'OUTPUTS'
PLOTS_DIR  = 'PLOTS'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR,  exist_ok=True)
print('Setup complete')

Setup complete


## 1 — Load Data

In [2]:
from features_task3 import load_raw, build_course_features, build_student_features

si, vle_meta, ass, st_ass, courses, arch, alerts = load_raw(DATA_DIR)
vle = pd.read_csv(f'{DATA_DIR}/studentVle.csv')
print(f'studentInfo:  {len(si):,} rows   studentVle: {len(vle):,} rows')

cf = build_course_features(vle_meta, ass, courses, si, DATA_DIR)
sf = build_student_features(si, st_ass, ass, arch, alerts)

display(cf[['code_module','historical_pass_rate','historical_withdrawal_rate',
            'n_assessments','module_presentation_length']]
        .groupby('code_module').mean().round(3))

Loading raw CSVs...


Done.


studentInfo:  32,593 rows   studentVle: 10,655,280 rows
Course features: 22 rows × 31 columns


Student features: 28785 rows × 20 columns


,historical_pass_rate,historical_withdrawal_rate,n_assessments,module_presentation_length
code_module,,,,
AAA,0.709,0.169,5.00,268.500
BBB,0.472,0.301,9.50,251.000
CCC,0.374,0.447,8.00,255.000
DDD,0.412,0.360,7.75,251.000
EEE,0.556,0.246,4.00,259.333
FFF,0.468,0.305,12.00,254.500
GGG,0.596,0.119,9.00,257.000


## 2 — VLE Profiles + Nearest Neighbours

In [3]:
from cf_vle import build_vle_profiles, compute_neighbours

profiles, click_weights, act_cols = build_vle_profiles(vle, vle_meta, si)
neighbours = compute_neighbours(profiles, click_weights, top_n=20)
print(f'Profile matrix: {profiles.shape}   Median top-1 sim: {neighbours["sim_0"].median():.4f}')

# VLE profiles by archetype
arch_map  = {6:'Steady Engager',5:'Recovering',4:'Coasting',3:'Anxious',
             2:'Struggling',1:'Early Dropout',0:'Ghost'}
norm_cols = [c for c in profiles.columns if c.startswith('norm_')]
prof_arch = profiles.join(sf.set_index('id_student')[['archetype_num']], how='inner')
prof_arch['archetype_label'] = prof_arch['archetype_num'].map(arch_map).fillna('Unknown')
arch_mean = prof_arch.groupby('archetype_label')[norm_cols[:8]].mean()

fig, ax = plt.subplots(figsize=(11, 4))
arch_mean.T.plot(ax=ax, marker='o')
ax.axhline(0, color='grey', lw=0.8, ls='--')
ax.set_title('VLE Activity Profile by Archetype (within-module normalised)')
ax.set_xlabel('Activity type')
ax.set_ylabel('Relative fraction vs module mean')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/vle_profiles_by_archetype.png', dpi=150)
plt.close()

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(neighbours['sim_0'].dropna(), bins=50, color='#4C72B0', edgecolor='white')
ax.set_title('Top-1 Pearson Similarity Distribution')
ax.set_xlabel('Similarity')
ax.set_ylabel('Students')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/similarity_distribution.png', dpi=150)
plt.close()
print('Saved VLE profile and similarity plots')

Merging VLE with activity types...


Computing per-student per-module activity-type profiles...


VLE profiles built: 26,074 students × 20 activity dims
Computing nearest neighbours (Pearson profile similarity)...


Neighbours computed: 26,074 students, top 20 each
Profile matrix: (26074, 20)   Median top-1 sim: 0.9927


Saved VLE profile and similarity plots


## 3 — Markov Transition Matrix

In [4]:
from transition import build_transition_matrix

trans_df, marg_df = build_transition_matrix(si)

pivot = (
    marg_df.groupby(['current_module','next_module'])['prob']
    .mean().reset_index()
    .pivot(index='current_module', columns='next_module', values='prob')
    .fillna(0)
)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(pivot.values, aspect='auto', cmap='Blues')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
ax.set_yticks(range(len(pivot.index)));  ax.set_yticklabels(pivot.index)
plt.colorbar(im, ax=ax, label='P(next | current)')
ax.set_title('Module Transition Probabilities P(next | current)')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/transition_heatmap.png', dpi=150)
plt.close()
print('Saved transition_heatmap.png')

Transition matrix built: 3,538 students, 3,808 transitions
Top 5 transitions:
current_module next_module   n
           CCC         EEE 611
           DDD         CCC 500
           EEE         CCC 497
           DDD         DDD 398
           FFF         FFF 349


Saved transition_heatmap.png


## 4 — Holdout Setup + Weight Tuning

In [5]:
from evaluate_task3 import build_holdout
from hybrid import grid_search_weights

holdout_students, already_taken_map, ground_truth_map = build_holdout(si)

rng      = np.random.default_rng(42)
tune_ids = rng.choice(holdout_students, size=max(1, int(len(holdout_students)*0.3)),
                      replace=False).tolist()
eval_ids = [s for s in holdout_students if s not in set(tune_ids)]
print(f'Holdout: {len(holdout_students):,}   Tune: {len(tune_ids):,}   Eval: {len(eval_ids):,}')

best_weights, gs_df = grid_search_weights(
    tune_ids, already_taken_map, sf, cf,
    neighbours, si, click_weights,
    trans_df, marg_df,
    {s: ground_truth_map[s] for s in tune_ids if s in ground_truth_map},
    top_k=3, step=0.1,
)

gs_df.head(10).to_csv(f'{OUTPUT_DIR}/weight_grid_search.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 4))
top10  = gs_df.head(10)
labels = [f"c={r.w_content}/m={r.w_markov}/f={r.w_cf}" for _, r in top10.iterrows()]
ax.barh(labels[::-1], top10['precision_at_k'].values[::-1], color='#4C72B0')
ax.set_xlabel('Precision@3')
ax.set_title('Top-10 Weight Combinations')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/weight_grid_search.png', dpi=150)
plt.close()
print(f'Best weights: {best_weights}')

Holdout: 1,113 students with observed next-module ground truth
Holdout: 1,113   Tune: 333   Eval: 780


Best weights: {'content': np.float64(0.0), 'markov': np.float64(1.0), 'cf': np.float64(0.0)}  =>  Precision@3 = 0.2783
Best weights: {'content': np.float64(0.0), 'markov': np.float64(1.0), 'cf': np.float64(0.0)}


## 5 — Generate + Evaluate Recommendations (Returning Students)

In [6]:
from hybrid import recommend_all
from evaluate_task3 import run_evaluation, method_breakdown

recs_df = recommend_all(
    eval_ids, already_taken_map, sf, cf,
    neighbours, si, click_weights,
    trans_df, marg_df,
    weights_hybrid=best_weights, top_k=3,
)
recs_df.to_csv(f'{OUTPUT_DIR}/recommendations.csv', index=False)
print(f'{len(recs_df):,} recommendations for {recs_df["id_student"].nunique():,} students')
display(recs_df.head(6))

gt_eval = {s: ground_truth_map[s] for s in eval_ids if s in ground_truth_map}
summary = run_evaluation(
    recs_df, eval_ids, already_taken_map, gt_eval,
    si, cf,
    all_modules=set(si['code_module'].unique()),
    top_k=3, output_dir=OUTPUT_DIR, plots_dir=PLOTS_DIR,
)
display(summary)
method_breakdown(recs_df, output_dir=OUTPUT_DIR, plots_dir=PLOTS_DIR)

2,340 recommendations for 780 students


,id_student,rank,module,hybrid_score,content_score,markov_score,cf_score,method_used,explanation
0,29411,1,EEE,1.0000,1.0000,1.0000,0.1257,Content+Markov+CF,EEE: matches your academic profile; commonly c...
1,29411,2,CCC,0.4458,0.0000,0.4458,0.8929,Markov+CF,CCC: commonly chosen next by students with sim...
2,29411,3,FFF,0.4458,0.9777,0.4458,1.0000,Content+Markov+CF,FFF: matches your academic profile; commonly c...
3,35355,1,CCC,1.0000,0.8833,1.0000,0.6854,Content+Markov+CF,CCC: matches your academic profile; commonly c...
4,35355,2,FFF,1.0000,0.8977,1.0000,1.0000,Content+Markov+CF,FFF: matches your academic profile; commonly c...
5,35355,3,DDD,0.9444,0.8556,0.9444,0.0000,Content+Markov,DDD: matches your academic profile; commonly c...


Hybrid        P@3=0.2705  Coverage=1.0000  Success=0.8013
Popularity    P@3=0.2145  Coverage=0.5714  Success=0.6359
PassRate      P@3=0.0192  Coverage=0.5714  Success=0.0577
Random        P@3=0.1688  Coverage=1.0000  Success=0.5026


Saved PLOTS/evaluation_comparison.png
Saved PLOTS/coverage_heatmap.png


,Method,Precision@3,Coverage,SuccessRate
0,Hybrid,0.2705,1.0000,0.8013
1,Popularity,0.2145,0.5714,0.6359
2,PassRate,0.0192,0.5714,0.0577
3,Random,0.1688,1.0000,0.5026



Method contribution breakdown:
  Markov      : 2,340 (100.0%)
  Content     : 1,626 (69.5%)
  CF          : 1,208 (51.6%)


## 6 — Cold-Start Evaluation (Brand-New 2014 Students)

In [7]:
from evaluate_task3 import build_coldstart_holdout, evaluate_coldstart

cs_students, cs_taken_map, cs_gt_map = build_coldstart_holdout(si)

cs_summary = evaluate_coldstart(
    cf, si, cs_students, cs_taken_map, cs_gt_map,
    top_k=3, output_dir=OUTPUT_DIR, plots_dir=PLOTS_DIR,
)
display(cs_summary)

Cold-start holdout: 340 brand-new students with observed next-module



Cold-start evaluation (340 students):
  Content cold-start:  P@3=0.0176  Success=0.0529
  Popularity baseline: P@3=0.2196  Success=0.6588
Saved PLOTS/coldstart_evaluation.png


,Method,Precision@3,SuccessRate
0,Cold-Content,0.0176,0.0529
1,Popularity,0.2196,0.6588


## 7 — Cold-Start Demo (Illustrative Profiles)

In [8]:
from content_based import cold_start_recommend

profiles_demo = {
    'Standard new student': {
        'imd_band_num': 5.0, 'age_band_num': 0, 'edu_num': 2,
        'studied_credits': 60, 'num_of_prev_attempts': 0,
        'prior_pass_rate': 0.5, 'risk_prob': 0.0,
        'archetype_num': 3.0, 'prior_dropout_flag': 0.0,
    },
    'Strong prior record': {
        'imd_band_num': 7.0, 'age_band_num': 1, 'edu_num': 4,
        'studied_credits': 120, 'num_of_prev_attempts': 0,
        'prior_pass_rate': 0.9, 'risk_prob': 0.0,
        'archetype_num': 6.0, 'prior_dropout_flag': 0.0,
    },
    'High risk (override active)': {
        'imd_band_num': 2.0, 'age_band_num': 0, 'edu_num': 1,
        'studied_credits': 60, 'num_of_prev_attempts': 2,
        'prior_pass_rate': 0.3, 'risk_prob': 0.82,
        'archetype_num': 1.0, 'prior_dropout_flag': 1.0,
    },
}

for label, profile in profiles_demo.items():
    recs = cold_start_recommend(profile, cf, top_k=3)
    print(f'\n{label}:')
    for r in recs:
        pr = cf[cf['code_module']==r['module']]['historical_pass_rate'].mean()
        print(f"  {r['module']}  score={r['score']:.4f}  hist_pass_rate={pr:.2f}  [{r['method']}]")


Standard new student:
  AAA  score=1.0000  hist_pass_rate=0.71  [Cold-Popularity]
  GGG  score=0.8402  hist_pass_rate=0.60  [Cold-Popularity]
  EEE  score=0.7842  hist_pass_rate=0.56  [Cold-Popularity]

Strong prior record:
  AAA  score=1.0000  hist_pass_rate=0.71  [Cold-Popularity]
  GGG  score=0.8402  hist_pass_rate=0.60  [Cold-Popularity]
  EEE  score=0.7842  hist_pass_rate=0.56  [Cold-Popularity]

High risk (override active):
  AAA  score=1.0000  hist_pass_rate=0.71  [Cold-Popularity-RiskFiltered]
  GGG  score=0.8402  hist_pass_rate=0.60  [Cold-Popularity-RiskFiltered]
  EEE  score=0.7842  hist_pass_rate=0.56  [Cold-Popularity-RiskFiltered]


## 8 — Individual Student Examples

In [9]:
from hybrid import hybrid_recommend

sf_idx = sf.set_index('id_student')
examples = {
    'Multi-module':  next((s for s in eval_ids if sf_idx.loc[s,'n_modules_taken']>=2), None),
    'Single-module': next((s for s in eval_ids if sf_idx.loc[s,'n_modules_taken']==1), None),
    'High-risk':     next((s for s in eval_ids if sf_idx.loc[s,'risk_prob']>=0.6), None),
}

for label, sid in examples.items():
    if sid is None: continue
    taken = already_taken_map.get(sid, set())
    gt    = ground_truth_map.get(sid, set())
    recs  = hybrid_recommend(
        sid, taken, sf, cf, neighbours, si, click_weights,
        trans_df, marg_df, weights_hybrid=best_weights, top_k=3
    )
    row = sf_idx.loc[sid]
    print(f'\n--- {label} (id={sid}) ---')
    print(f'  History: {taken}   GT: {gt}   Risk: {row.get("risk_prob",0):.3f}   Last: {row.get("last_module")} / {row.get("last_outcome")}')
    for r in recs:
        hit = 'HIT' if r['module'] in gt else '   '
        print(f'  [{hit}] {r["module"]}  hybrid={r["hybrid_score"]:.4f}  c={r["content_score"]:.3f} m={r["markov_score"]:.3f} cf={r["cf_score"]:.3f}  ({r["method_used"]})')
        print(f'         {r["explanation"]}')


--- Multi-module (id=29411) ---
  History: {'DDD'}   GT: {'CCC'}   Risk: 0.872   Last: CCC / Withdrawn
  [   ] EEE  hybrid=1.0000  c=1.000 m=1.000 cf=0.126  (Content+Markov+CF)
         EEE: matches your academic profile; commonly chosen next by students with similar outcomes.
  [HIT] CCC  hybrid=0.4458  c=0.000 m=0.446 cf=0.893  (Markov+CF)
         CCC: commonly chosen next by students with similar outcomes; popular among students with similar study patterns.
  [   ] FFF  hybrid=0.4458  c=0.978 m=0.446 cf=1.000  (Content+Markov+CF)
         FFF: matches your academic profile; commonly chosen next by students with similar outcomes; popular among students with similar study patterns.

--- High-risk (id=29411) ---
  History: {'DDD'}   GT: {'CCC'}   Risk: 0.872   Last: CCC / Withdrawn
  [   ] EEE  hybrid=1.0000  c=1.000 m=1.000 cf=0.126  (Content+Markov+CF)
         EEE: matches your academic profile; commonly chosen next by students with similar outcomes.
  [HIT] CCC  hybrid=0.4458  c=

## 9 — Summary

In [10]:
print('=' * 62)
print('TASK 3 — FINAL SUMMARY')
print('=' * 62)
print(f'  {si["id_student"].nunique():,} students, 7 modules, 22 course-presentations')
print(f'  87.7% took only 1 module — standard CF unusable for majority')
print()
print('  Signals:')
print('    Markov  : P(next | current, outcome)  +  Laplace smoothing + marginal fallback')
print('    VLE-CF  : Pearson on within-module activity profiles, top-20 neighbours')
print('    Content : cosine on 6-dim student–course feature space')
print()
print(f'  Best weights (grid search): {best_weights}')
print()
print('  Returning students (2013->2014):')
for _, row in summary.iterrows():
    print(f'    {row["Method"]:12s}  P@3={row.iloc[1]:.4f}  Coverage={row["Coverage"]:.4f}  Success={row["SuccessRate"]:.4f}')
print()
print('  Cold-start (brand-new 2014 students):')
for _, row in cs_summary.iterrows():
    print(f'    {row["Method"]:16s}  P@3={row.iloc[1]:.4f}  Success={row["SuccessRate"]:.4f}')
print()
print('Files:')
for d in [OUTPUT_DIR, PLOTS_DIR]:
    for f in sorted(glob.glob(f'{d}/*')):
        print(f'  [{d}] {os.path.basename(f)}')

TASK 3 — FINAL SUMMARY
  28,785 students, 7 modules, 22 course-presentations
  87.7% took only 1 module — standard CF unusable for majority

  Signals:
    Markov  : P(next | current, outcome)  +  Laplace smoothing + marginal fallback
    VLE-CF  : Pearson on within-module activity profiles, top-20 neighbours
    Content : cosine on 6-dim student–course feature space

  Best weights (grid search): {'content': np.float64(0.0), 'markov': np.float64(1.0), 'cf': np.float64(0.0)}

  Returning students (2013->2014):
    Hybrid        P@3=0.2705  Coverage=1.0000  Success=0.8013
    Popularity    P@3=0.2145  Coverage=0.5714  Success=0.6359
    PassRate      P@3=0.0192  Coverage=0.5714  Success=0.0577
    Random        P@3=0.1688  Coverage=1.0000  Success=0.5026

  Cold-start (brand-new 2014 students):
    Cold-Content      P@3=0.0176  Success=0.0529
    Popularity        P@3=0.2196  Success=0.6588

Files:
  [OUTPUTS] coldstart_evaluation.csv
  [OUTPUTS] coverage_heatmap.png
  [OUTPUTS] evaluat